<a href="https://colab.research.google.com/github/bbrotwurst-tech/NBA_NCAA-predictor/blob/main/NBA_predictor_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# NBA PREDICTION MODEL - SETUP
# ============================================

!pip install nba_api -q
!pip install xgboost -q

import pandas as pd
import numpy as np
from nba_api.stats.endpoints import leaguegamefinder, leaguedashteamstats
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 5.8 MB/s eta 0:00:00
✅ All packages ready


In [ ]:
import time
from nba_api.stats.library.http import NBAStatsHTTP

# Bump the default timeout used by nba_api's HTTP client
NBAStatsHTTP.headers = {
    **NBAStatsHTTP.headers,
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com',
}

def fetch_season_games(season, retries=5, delay=15):
    for attempt in range(retries):
        try:
            gamefinder = leaguegamefinder.LeagueGameFinder(
                season_nullable=season,
                league_id_nullable='00',
                timeout=90
            )
            return gamefinder.get_data_frames()[0]
        except Exception as e:
            print(f"   Attempt {attempt+1}/{retries} failed for {season}: {type(e).__name__}")
            if attempt < retries - 1:
                time.sleep(delay)
    raise RuntimeError(f"Failed to fetch {season} after {retries} attempts")

seasons = ['2023-24', '2024-25', '2025-26']
all_games_list = []
for s in seasons:
    print(f"Fetching {s}...")
    all_games_list.append(fetch_season_games(s))
    time.sleep(3)  # small gap between seasons too

all_games = pd.concat(all_games_list, ignore_index=True)
all_games = all_games[all_games['SEASON_ID'].str[0] == '2']
print(f"Loaded {len(all_games)} games")

Fetching 2023-24...
   Attempt 1/5 failed for 2023-24: ReadTimeout
   Attempt 2/5 failed for 2023-24: ReadTimeout
   Attempt 3/5 failed for 2023-24: ReadTimeout
   Attempt 4/5 failed for 2023-24: ReadTimeout
   Attempt 5/5 failed for 2023-24: ReadTimeout


RuntimeError: Failed to fetch 2023-24 after 5 attempts

In [ ]:
# ============================================
# FULL MODEL PIPELINE
# ============================================

print("=" * 60)
print("REBUILDING FULL MODEL PIPELINE")
print("=" * 60)

# STEP 1: Fetch Data
print("\n[1/5] Fetching NBA data...")
def fetch_season_games(season):
    gamefinder = leaguegamefinder.LeagueGameFinder(
        season_nullable=season,
        league_id_nullable='00'
    )
    return gamefinder.get_data_frames()[0]

seasons = ['2023-24', '2024-25', '2025-26']
all_games = pd.concat([fetch_season_games(s) for s in seasons], ignore_index=True)
all_games = all_games[all_games['SEASON_ID'].str[0] == '2']
print(f"   Loaded {len(all_games)} games")

# STEP 2: Structure Matchups
print("\n[2/5] Structuring matchups...")
home_games = all_games[all_games['MATCHUP'].str.contains('vs.')].copy()
away_games = all_games[all_games['MATCHUP'].str.contains('@')].copy()

home_cols = {col: f'HOME_{col}' for col in home_games.columns if col not in ['GAME_ID', 'GAME_DATE']}
away_cols = {col: f'AWAY_{col}' for col in away_games.columns if col not in ['GAME_ID', 'GAME_DATE']}

home_games.rename(columns=home_cols, inplace=True)
away_games.rename(columns=away_cols, inplace=True)

model_data = pd.merge(home_games, away_games, on=['GAME_ID', 'GAME_DATE'], how='inner')
model_data['GAME_DATE'] = pd.to_datetime(model_data['GAME_DATE'])
model_data = model_data.sort_values('GAME_DATE').reset_index(drop=True)

model_data['HOME_MARGIN'] = model_data['HOME_PTS'] - model_data['AWAY_PTS']
model_data['HOME_WIN'] = (model_data['HOME_MARGIN'] > 0).astype(int)
print(f"   {len(model_data)} unique matchups")

# STEP 3: Calculate Elo Ratings
print("\n[3/5] Calculating Elo ratings...")
def calculate_elo(df, k_factor=20):
    elo_dict = {}
    home_elo_list = []
    away_elo_list = []
    for idx, row in df.iterrows():
        home = row['HOME_TEAM_ID']
        away = row['AWAY_TEAM_ID']
        if home not in elo_dict: elo_dict[home] = 1500
        if away not in elo_dict: elo_dict[away] = 1500
        home_elo = elo_dict[home]
        away_elo = elo_dict[away]
        home_elo_list.append(home_elo)
        away_elo_list.append(away_elo)
        expected_home = 1 / (1 + 10 ** ((away_elo - home_elo - 100) / 400))
        margin = row['HOME_MARGIN']
        if margin > 0:
            actual_home = 1
            margin_multiplier = np.log(abs(margin) + 1) * 2
        elif margin < 0:
            actual_home = 0
            margin_multiplier = np.log(abs(margin) + 1) * 2
        else:
            actual_home = 0.5
            margin_multiplier = 1
        elo_change = k_factor * margin_multiplier * (actual_home - expected_home)
        elo_dict[home] += elo_change
        elo_dict[away] -= elo_change
    df['HOME_ELO'] = home_elo_list
    df['AWAY_ELO'] = away_elo_list
    df['ELO_DIFF'] = df['HOME_ELO'] - df['AWAY_ELO']
    return df

model_data = calculate_elo(model_data)
print(f"   Elo range: {model_data['HOME_ELO'].min():.0f} to {model_data['HOME_ELO'].max():.0f}")

# STEP 4: Feature Engineering
print("\n[4/5] Building features...")
model_data['ELO_DIFF_HOME_ADJ'] = model_data['ELO_DIFF'] + 100
model_data['HOME_REST'] = model_data.groupby('HOME_TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(3)
model_data['AWAY_REST'] = model_data.groupby('AWAY_TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(3)
model_data['REST_ADVANTAGE_SIGNED'] = model_data['HOME_REST'] - model_data['AWAY_REST']
model_data['HOME_ELO_MOMENTUM'] = model_data.groupby('HOME_TEAM_ID')['HOME_ELO'].diff(5).fillna(0)
model_data['AWAY_ELO_MOMENTUM'] = model_data.groupby('AWAY_TEAM_ID')['AWAY_ELO'].diff(5).fillna(0)

# Four Factors (league averages as fallback)
print("   Using league average Four Factors...")
model_data['EFG_DIFF'] = 0.0
model_data['TOV_DIFF'] = 0.0
model_data['FT_RATE_DIFF'] = 0.0

win_features = [
    'ELO_DIFF', 'ELO_DIFF_HOME_ADJ', 'HOME_ELO', 'AWAY_ELO',
    'REST_ADVANTAGE_SIGNED', 'HOME_ELO_MOMENTUM', 'AWAY_ELO_MOMENTUM',
    'EFG_DIFF', 'TOV_DIFF', 'FT_RATE_DIFF'
]
print(f"   {len(win_features)} features created")

# STEP 5: Train Model
print("\n[5/5] Training Random Forest...")
split_date = pd.to_datetime('2025-01-01')
train_mask = model_data['GAME_DATE'] < split_date
test_mask = model_data['GAME_DATE'] >= split_date

X_train = model_data.loc[train_mask, win_features].fillna(0)
X_test = model_data.loc[test_mask, win_features].fillna(0)
y_train = model_data.loc[train_mask, 'HOME_WIN']
y_test = model_data.loc[test_mask, 'HOME_WIN']

rf = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_split=20,
    min_samples_leaf=10, random_state=42, class_weight='balanced'
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)
print(f"Train games: {len(X_train)}")
print(f"Test games: {len(X_test)}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

print("\n✅ Full pipeline rebuilt successfully!")

REBUILDING FULL MODEL PIPELINE

[1/5] Fetching NBA data...


ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
# ============================================
# SPREAD PREDICTION MODEL
# ============================================

print("=" * 60)
print("TRAINING SPREAD PREDICTION MODEL")
print("=" * 60)

y_margin = model_data['HOME_MARGIN']

split_date = pd.to_datetime('2025-01-01')
train_mask = model_data['GAME_DATE'] < split_date
test_mask = model_data['GAME_DATE'] >= split_date

X_train_spread = model_data.loc[train_mask, win_features].fillna(0)
X_test_spread = model_data.loc[test_mask, win_features].fillna(0)
y_train_spread = y_margin[train_mask]
y_test_spread = y_margin[test_mask]

rf_spread = RandomForestRegressor(
    n_estimators=150, max_depth=6, min_samples_split=20,
    min_samples_leaf=10, random_state=42, n_jobs=-1
)
rf_spread.fit(X_train_spread, y_train_spread)

y_pred_spread = rf_spread.predict(X_test_spread)
mae = mean_absolute_error(y_test_spread, y_pred_spread)
within_5 = (abs(y_test_spread - y_pred_spread) <= 5).mean()

print(f"\n--- Spread Model Performance ---")
print(f"MAE: {mae:.2f} points")
print(f"Within 5 points: {within_5:.1%}")
print("✅ Spread model ready!")

TRAINING SPREAD PREDICTION MODEL

--- Spread Model Performance ---
MAE: 11.56 points
Within 5 points: 29.1%
✅ Spread model ready!


In [ ]:
# ============================================
# POINT TOTALS PREDICTION MODEL
# ============================================

print("=" * 60)
print("TRAINING POINT TOTALS MODEL")
print("=" * 60)

model_data['TOTAL_PTS'] = model_data['HOME_PTS'] + model_data['AWAY_PTS']
model_data['HOME_OFF_RTG'] = model_data['HOME_PTS'] / (model_data['HOME_FGA'] + 0.44 * model_data['HOME_FTA'] + model_data['HOME_TOV']) * 100
model_data['AWAY_OFF_RTG'] = model_data['AWAY_PTS'] / (model_data['AWAY_FGA'] + 0.44 * model_data['AWAY_FTA'] + model_data['AWAY_TOV']) * 100

model_data['HOME_PACE'] = model_data['HOME_FGA'] + 0.44 * model_data['HOME_FTA'] + model_data['HOME_TOV']
model_data['AWAY_PACE'] = model_data['AWAY_FGA'] + 0.44 * model_data['AWAY_FTA'] + model_data['AWAY_TOV']
model_data['AVG_PACE'] = (model_data['HOME_PACE'] + model_data['AWAY_PACE']) / 2

model_data['HOME_ORTG_ROLL5'] = model_data.groupby('HOME_TEAM_ID')['HOME_OFF_RTG'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
model_data['AWAY_ORTG_ROLL5'] = model_data.groupby('AWAY_TEAM_ID')['AWAY_OFF_RTG'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
model_data['HOME_TOTAL_ROLL5'] = model_data.groupby('HOME_TEAM_ID')['TOTAL_PTS'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
model_data['AWAY_TOTAL_ROLL5'] = model_data.groupby('AWAY_TEAM_ID')['TOTAL_PTS'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())

totals_features = ['AVG_PACE', 'HOME_ORTG_ROLL5', 'AWAY_ORTG_ROLL5', 'HOME_TOTAL_ROLL5', 'AWAY_TOTAL_ROLL5', 'ELO_DIFF']

split_date = pd.to_datetime('2025-01-01')
train_mask = model_data['GAME_DATE'] < split_date
test_mask = model_data['GAME_DATE'] >= split_date

X_train_totals = model_data.loc[train_mask, totals_features].fillna(0)
X_test_totals = model_data.loc[test_mask, totals_features].fillna(0)
y_train_totals = model_data.loc[train_mask, 'TOTAL_PTS']
y_test_totals = model_data.loc[test_mask, 'TOTAL_PTS']

rf_totals = RandomForestRegressor(n_estimators=150, max_depth=6, min_samples_split=20, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf_totals.fit(X_train_totals, y_train_totals)

y_pred_totals = rf_totals.predict(X_test_totals)
mae_totals = mean_absolute_error(y_test_totals, y_pred_totals)

print(f"\n--- Totals Model Performance ---")
print(f"MAE: {mae_totals:.2f} points")
print(f"Within 5 points: {(abs(y_test_totals - y_pred_totals) <= 5).mean():.1%}")
print("✅ Totals model ready!")

TRAINING POINT TOTALS MODEL

--- Totals Model Performance ---
MAE: 14.23 points
Within 5 points: 22.1%
✅ Totals model ready!


In [ ]:
# ============================================
# PLAYER VALUE DATABASE
# ============================================

PLAYER_VALUES = {
    'Nikola Jokic': 10.0, 'Shai Gilgeous-Alexander': 9.5, 'Giannis Antetokounmpo': 9.0,
    'Luka Doncic': 9.0, 'Jayson Tatum': 8.5, 'Joel Embiid': 8.5, 'Stephen Curry': 8.0,
    'Kevin Durant': 7.5, 'LeBron James': 7.5, 'Anthony Davis': 7.0, 'Devin Booker': 7.0,
    'Anthony Edwards': 7.0, 'Victor Wembanyama': 7.0, 'Jalen Brunson': 6.5,
    'Donovan Mitchell': 6.5, 'Jaylen Brown': 5.5, 'Karl-Anthony Towns': 5.0,
    'Cade Cunningham': 4.5, 'Paolo Banchero': 4.5, 'Franz Wagner': 4.0,
    'Jalen Williams': 4.5, 'Chet Holmgren': 4.5, 'Evan Mobley': 4.5,
    'Jaren Jackson Jr.': 4.5, 'Trae Young': 6.0, 'Tyrese Haliburton': 6.0,
    'Bam Adebayo': 6.0, 'Ja Morant': 6.0, 'Zion Williamson': 6.5,
    'Damian Lillard': 5.5, 'Kyrie Irving': 5.5, 'De\'Aaron Fox': 5.0,
    'Jamal Murray': 4.5, 'Kristaps Porzingis': 3.5, 'Draymond Green': 3.5,
    'CJ McCollum': 3.5, 'DeMar DeRozan': 3.5, 'Bradley Beal': 3.5,
}
DEFAULT_PLAYER_VALUE = 2.0

def get_player_value(player_name):
    return PLAYER_VALUES.get(player_name, DEFAULT_PLAYER_VALUE)

print(f"✅ Player value database loaded ({len(PLAYER_VALUES)} players)")

✅ Player value database loaded (38 players)


In [ ]:
# ============================================
# TEAM INJURY IMPACT CALCULATOR
# ============================================

TEAM_STARS = {
    'ATL': ['Trae Young', 'Jalen Johnson', 'Dyson Daniels'],
    'BOS': ['Jayson Tatum', 'Jaylen Brown', 'Kristaps Porzingis', 'Jrue Holiday'],
    'BKN': ['Cameron Johnson', 'Nic Claxton'],
    'CHA': ['LaMelo Ball', 'Brandon Miller'],
    'CHI': ['Coby White', 'Nikola Vucevic'],
    'CLE': ['Donovan Mitchell', 'Darius Garland', 'Evan Mobley', 'Jarrett Allen'],
    'DAL': ['Anthony Davis', 'Kyrie Irving'],
    'DEN': ['Nikola Jokic', 'Jamal Murray', 'Aaron Gordon'],
    'DET': ['Cade Cunningham', 'Jaden Ivey'],
    'GSW': ['Stephen Curry', 'Jimmy Butler', 'Draymond Green'],
    'HOU': ['Alperen Sengun', 'Jalen Green'],
    'IND': ['Tyrese Haliburton', 'Pascal Siakam', 'Myles Turner'],
    'LAC': ['Kawhi Leonard', 'James Harden'],
    'LAL': ['LeBron James', 'Luka Doncic'],
    'MEM': ['Ja Morant', 'Jaren Jackson Jr.'],
    'MIA': ['Bam Adebayo', 'Tyler Herro'],
    'MIL': ['Giannis Antetokounmpo', 'Damian Lillard'],
    'MIN': ['Anthony Edwards', 'Rudy Gobert'],
    'NOP': ['Zion Williamson', 'CJ McCollum'],
    'NYK': ['Jalen Brunson', 'Karl-Anthony Towns', 'Mikal Bridges', 'OG Anunoby'],
    'OKC': ['Shai Gilgeous-Alexander', 'Jalen Williams', 'Chet Holmgren'],
    'ORL': ['Paolo Banchero', 'Franz Wagner'],
    'PHI': ['Joel Embiid', 'Tyrese Maxey', 'Paul George'],
    'PHX': ['Devin Booker', 'Kevin Durant'],
    'POR': ['Anfernee Simons', 'Shaedon Sharpe', 'Deandre Ayton'],
    'SAC': ['Domantas Sabonis', 'DeMar DeRozan', 'Zach LaVine'],
    'SAS': ['Victor Wembanyama', 'De\'Aaron Fox'],
    'TOR': ['Scottie Barnes', 'Immanuel Quickley'],
    'UTA': ['Lauri Markkanen', 'Collin Sexton'],
    'WAS': ['Jordan Poole', 'Kyle Kuzma'],
}

def calculate_injury_impact(team_abbr, missing_players_list):
    if not missing_players_list:
        return 0.0
    total_impact = 0.0
    for player in missing_players_list:
        impact = get_player_value(player)
        total_impact += impact
        print(f"      {player}: -{impact:.1f}%")
    return total_impact

def adjust_for_injuries_weighted(base_prob, home_team, away_team, home_missing=None, away_missing=None):
    if home_missing is None: home_missing = []
    if away_missing is None: away_missing = []
    print("\n" + "-" * 40)
    print("WEIGHTED INJURY IMPACT")
    print("-" * 40)
    print(f"\n{home_team} missing:")
    home_impact = calculate_injury_impact(home_team, home_missing)
    print(f"\n{away_team} missing:")
    away_impact = calculate_injury_impact(away_team, away_missing)
    net_adjustment = away_impact - home_impact
    adjusted_prob = base_prob + (net_adjustment / 100)
    adjusted_prob = max(0.05, min(0.95, adjusted_prob))
    print(f"\n--- Summary ---")
    print(f"Base: {base_prob:.1%} → Adjusted: {adjusted_prob:.1%}")
    return adjusted_prob

print("✅ Injury calculator ready")

✅ Injury calculator ready


In [ ]:
# ============================================
# DAILY GAME PREDICTION
# ============================================

import pandas as pd
import numpy as np

try:
    _ = model_data.shape
    print("✅ model_data found")
except NameError:
    print("⚠️ Run Cell 2 first!")

def predict_game_raw(home_team_abbr, away_team_abbr):
    home_id = model_data[model_data['HOME_TEAM_ABBREVIATION'] == home_team_abbr]['HOME_TEAM_ID'].iloc[-1]
    away_id = model_data[model_data['AWAY_TEAM_ABBREVIATION'] == away_team_abbr]['AWAY_TEAM_ID'].iloc[-1]
    home_elo = model_data[model_data['HOME_TEAM_ID'] == home_id]['HOME_ELO'].iloc[-1]
    away_elo = model_data[model_data['AWAY_TEAM_ID'] == away_id]['AWAY_ELO'].iloc[-1]
    features = pd.DataFrame([{
        'ELO_DIFF': home_elo - away_elo,
        'ELO_DIFF_HOME_ADJ': home_elo - away_elo + 100,
        'HOME_ELO': home_elo, 'AWAY_ELO': away_elo,
        'REST_ADVANTAGE_SIGNED': 0, 'HOME_ELO_MOMENTUM': 0,
        'AWAY_ELO_MOMENTUM': 0, 'EFG_DIFF': 0, 'TOV_DIFF': 0, 'FT_RATE_DIFF': 0
    }])
    proba = rf.predict_proba(features[win_features])[0, 1]
    return proba, home_elo, away_elo

# CHANGE THESE
HOME_TEAM = 'HOU'
AWAY_TEAM = 'LAL'

try:
    raw_prob, home_elo, away_elo = predict_game_raw(HOME_TEAM, AWAY_TEAM)

    print("=" * 60)
    print(f"MATCHUP: {AWAY_TEAM} @ {HOME_TEAM}")
    print("=" * 60)
    print(f"\nElo Ratings:")
    print(f"  {HOME_TEAM}: {home_elo:.0f}")
    print(f"  {AWAY_TEAM}: {away_elo:.0f}")
    print(f"  Difference: {home_elo - away_elo:+.0f}")
    print(f"\nBase Model Probability:")
    print(f"  {HOME_TEAM} win: {raw_prob:.1%}")
    print(f"  {AWAY_TEAM} win: {1 - raw_prob:.1%}")

    print("\n" + "-" * 60)
    print("INJURY ADJUSTMENT")
    print("-" * 60)
    print("\nEnter injured star player names (comma-separated, or press Enter for none)")

    home_input = input(f"\nStars OUT for {HOME_TEAM}: ")
    away_input = input(f"Stars OUT for {AWAY_TEAM}: ")

    home_missing = [p.strip() for p in home_input.split(',') if p.strip()]
    away_missing = [p.strip() for p in away_input.split(',') if p.strip()]

    adjusted_prob = adjust_for_injuries_weighted(raw_prob, HOME_TEAM, AWAY_TEAM, home_missing, away_missing)

    print("\n" + "=" * 60)
    print("FINAL PREDICTION")
    print("=" * 60)
    print(f"\n{HOME_TEAM} win probability: {adjusted_prob:.1%}")
    print(f"{AWAY_TEAM} win probability: {1 - adjusted_prob:.1%}")

    print("\n" + "-" * 60)
    print("BETTING ANALYSIS")
    print("-" * 60)
    market_implied = float(input("\nEnter market implied probability (e.g., 0.55 for 55%): ") or 0.50)

    if adjusted_prob > market_implied + 0.05:
        print(f"\n✅ EDGE: Model ({adjusted_prob:.1%}) > Market ({market_implied:.1%}) | Bet {HOME_TEAM}")
    elif (1 - adjusted_prob) > (1 - market_implied) + 0.05:
        print(f"\n✅ EDGE: Model ({1 - adjusted_prob:.1%}) > Market ({1 - market_implied:.1%}) | Bet {AWAY_TEAM}")
    else:
        print(f"\n❌ NO EDGE: Model and market too close")

except NameError as e:
    print(f"\n❌ ERROR: {e}")
    print("Run Cell 2 first!")

✅ model_data found
MATCHUP: LAL @ HOU

Elo Ratings:
  HOU: 1683
  LAL: 1651
  Difference: +32

Base Model Probability:
  HOU win: 55.7%
  LAL win: 44.3%

------------------------------------------------------------
INJURY ADJUSTMENT
------------------------------------------------------------

Enter injured star player names (comma-separated, or press Enter for none)


KeyboardInterrupt: Interrupted by user

In [ ]:
# ============================================
# SPREAD PREDICTION: ATL @ NYK
# ============================================

def predict_spread(home_team_abbr, away_team_abbr, home_stars_out=0, away_stars_out=0):
    home_id = model_data[model_data['HOME_TEAM_ABBREVIATION'] == home_team_abbr]['HOME_TEAM_ID'].iloc[-1]
    away_id = model_data[model_data['AWAY_TEAM_ABBREVIATION'] == away_team_abbr]['AWAY_TEAM_ID'].iloc[-1]

    home_elo = model_data[model_data['HOME_TEAM_ID'] == home_id]['HOME_ELO'].iloc[-1]
    away_elo = model_data[model_data['AWAY_TEAM_ID'] == away_id]['AWAY_ELO'].iloc[-1]

    features = pd.DataFrame([{
        'ELO_DIFF': home_elo - away_elo,
        'ELO_DIFF_HOME_ADJ': home_elo - away_elo + 100,
        'HOME_ELO': home_elo, 'AWAY_ELO': away_elo,
        'REST_ADVANTAGE_SIGNED': 0, 'HOME_ELO_MOMENTUM': 0,
        'AWAY_ELO_MOMENTUM': 0, 'EFG_DIFF': 0, 'TOV_DIFF': 0, 'FT_RATE_DIFF': 0
    }])

    predicted_margin = rf_spread.predict(features[win_features])[0]
    injury_adj = (away_stars_out - home_stars_out) * 3.0
    return predicted_margin + injury_adj

print("=" * 60)
print("SPREAD PREDICTION: ATL @ NYK")
print("=" * 60)

predicted_margin = predict_spread('NYK', 'ATL')

if predicted_margin > 0:
    print(f"\nPredicted margin: NYK by {predicted_margin:.1f} points")
    print(f"Implied spread: NYK -{predicted_margin:.1f}")
else:
    print(f"\nPredicted margin: ATL by {-predicted_margin:.1f} points")
    print(f"Implied spread: ATL -{-predicted_margin:.1f}")

# Context
print(f"\n--- Context ---")
print(f"Earlier prediction: ATL -4.0")
print(f"Market opened: NYK -1.5")
print(f"Look for Kalshi 'No' contracts on wide NYK margins")

SPREAD PREDICTION: ATL @ NYK

Predicted margin: NYK by 2.4 points
Implied spread: NYK -2.4

--- Context ---
Earlier prediction: ATL -4.0
Market opened: NYK -1.5
Look for Kalshi 'No' contracts on wide NYK margins


In [ ]:
# ============================================
# PREDICT SPECIFIC MATCHUP
# ============================================

def predict_matchup(home_team_abbr, away_team_abbr,
                    home_injured=None, away_injured=None,
                    market_prob=None):
    """
    Full prediction for any NBA matchup.

    Parameters:
    - home_team_abbr: Home team (e.g., 'BOS')
    - away_team_abbr: Away team (e.g., 'PHI')
    - home_injured: List of injured players for home team (e.g., ['Kristaps Porzingis'])
    - away_injured: List of injured players for away team (e.g., ['Joel Embiid'])
    - market_prob: Market implied probability (e.g., 0.55 for 55%)
    """
    if home_injured is None: home_injured = []
    if away_injured is None: away_injured = []

    # Get Elo ratings
    home_id = model_data[model_data['HOME_TEAM_ABBREVIATION'] == home_team_abbr]['HOME_TEAM_ID'].iloc[-1]
    away_id = model_data[model_data['AWAY_TEAM_ABBREVIATION'] == away_team_abbr]['AWAY_TEAM_ID'].iloc[-1]
    home_elo = model_data[model_data['HOME_TEAM_ID'] == home_id]['HOME_ELO'].iloc[-1]
    away_elo = model_data[model_data['AWAY_TEAM_ID'] == away_id]['AWAY_ELO'].iloc[-1]

    # Build features for models
    features = pd.DataFrame([{
        'ELO_DIFF': home_elo - away_elo,
        'ELO_DIFF_HOME_ADJ': home_elo - away_elo + 100,
        'HOME_ELO': home_elo, 'AWAY_ELO': away_elo,
        'REST_ADVANTAGE_SIGNED': 0, 'HOME_ELO_MOMENTUM': 0,
        'AWAY_ELO_MOMENTUM': 0, 'EFG_DIFF': 0, 'TOV_DIFF': 0, 'FT_RATE_DIFF': 0
    }])

    # --- MONEYLINE ---
    raw_prob = rf.predict_proba(features[win_features])[0, 1]

    # Weighted injury adjustment
    adjusted_prob = adjust_for_injuries_weighted(raw_prob, home_team_abbr, away_team_abbr,
                                                  home_injured, away_injured)

    # --- SPREAD ---
    predicted_margin = rf_spread.predict(features[win_features])[0]
    injury_margin_adj = (len(away_injured) - len(home_injured)) * 3.0
    spread = predicted_margin + injury_margin_adj

    # --- DISPLAY ---
    print("=" * 60)
    print(f"MATCHUP: {away_team_abbr} @ {home_team_abbr}")
    print("=" * 60)

    print(f"\n📊 ELO RATINGS")
    print(f"  {home_team_abbr}: {home_elo:.0f}  |  {away_team_abbr}: {away_elo:.0f}  |  Diff: {home_elo - away_elo:+.0f}")

    print(f"\n🎯 MONEYLINE PREDICTION")
    print(f"  {home_team_abbr} win: {adjusted_prob:.1%}")
    print(f"  {away_team_abbr} win: {1 - adjusted_prob:.1%}")

    if home_injured or away_injured:
        print(f"\n  📝 Injury adjustment: {raw_prob:.1%} → {adjusted_prob:.1%}")

    print(f"\n📏 SPREAD PREDICTION")
    if spread > 0:
        print(f"  {home_team_abbr} by {spread:.1f} points")
    else:
        print(f"  {away_team_abbr} by {-spread:.1f} points")

    # --- BETTING ANALYSIS ---
    if market_prob is not None:
        print(f"\n💰 BETTING ANALYSIS")
        print(f"  Model: {adjusted_prob:.1%}  |  Market: {market_prob:.1%}")

        home_edge = adjusted_prob - market_prob
        away_edge = (1 - adjusted_prob) - (1 - market_prob)

        if home_edge > 0.05:
            print(f"  ✅ Edge on {home_team_abbr}: +{home_edge:.1%}")
        elif away_edge > 0.05:
            print(f"  ✅ Edge on {away_team_abbr}: +{away_edge:.1%}")
        else:
            print(f"  ❌ No significant edge")

    return adjusted_prob, spread

# ============================================
# CHANGE THESE FOR YOUR GAME
# ============================================
HOME = 'SAS'        # Home team
AWAY = 'NYK'        # Away team
HOME_INJURED = ['David Jones Garcia']   # e.g., ['Kristaps Porzingis']
AWAY_INJURED = ['Mitchell ]   # e.g., ['Joel Embiid']
MARKET = None       # e.g., 0.73 for 73% implied (or None to skip)

prob, spread = predict_matchup(HOME, AWAY, HOME_INJURED, AWAY_INJURED, MARKET)


----------------------------------------
WEIGHTED INJURY IMPACT
----------------------------------------

SAS missing:
      David Jones Garcia: -2.0%

OKC missing:
      Ajay Mitchell: -2.0%
      Jalen Williams: -4.5%
      Thomas Sorber: -2.0%

--- Summary ---
Base: 54.7% → Adjusted: 61.2%
MATCHUP: OKC @ SAS

📊 ELO RATINGS
  SAS: 1955  |  OKC: 1989  |  Diff: -34

🎯 MONEYLINE PREDICTION
  SAS win: 61.2%
  OKC win: 38.8%

  📝 Injury adjustment: 54.7% → 61.2%

📏 SPREAD PREDICTION
  SAS by 8.2 points
